In [ ]:
import kagglehub
!pip install ultralytics
from ultralytics import YOLO
from IPython import display
from IPython.display import display as displayfilechooser
import matplotlib.pyplot as plt
from ipyfilechooser import FileChooser
from pathlib import Path
import pandas as pd
from collections import Counter
from google.colab import files

# Dataset Downloading and Display

In [ ]:
# Download latest version of street signs data set
data_path = kagglehub.dataset_download("pkdarabi/cardetection")

print("data set stored at:" + data_path)
data_yaml = f"{data_path}/car/data.yaml"

Dataset Summary:

In [ ]:
c = Counter()

# Class IDs from the dataset (based on model validation output)
signs = {
    0: 'Green Light',
    1: 'Red Light',
    2: "Speed Limit 10",
    3: 'Speed Limit 100',
    4: 'Speed Limit 110',
    5: 'Speed Limit 120',
    6: 'Speed Limit 20',
    7: 'Speed Limit 30',
    8: 'Speed Limit 40',
    9: 'Speed Limit 50',
    10: 'Speed Limit 60',
    11: 'Speed Limit 70',
    12: 'Speed Limit 80',
    13: 'Speed Limit 90',
    14: 'Stop',
}

for metadata_file in Path(data_path).rglob("*.txt"):
    content = metadata_file.read_text().strip()
    if content:
        for line in content.splitlines():
            line = line.strip()
            if not line or line.startswith('#'): #some files have a comment
                continue
            try:
                class_id = int(line.split()[0])
                c[class_id] += 1
            except ValueError:
                continue

print("Sign occurrences in dataset:\n")
for class_id, count in sorted(c.items()):
    name = signs.get(class_id, f"Unknown ({class_id})")
    print(f"{name}: {count}")

# Model Training

In [ ]:
model = YOLO('yolov8n.pt') #nano size model
data_yaml = f"{data_path}/car/data.yaml"
#just a safety check to prevent accidental time waste
response = input("About to train the model. This can take a very long time. Do you want to proceed? (y/n): ")
if response.lower() == 'y':
    # Train model off dataset
    results = model.train(data=data_yaml, epochs=50, imgsz=416)
else:
    print("Cancelled.")

# Model Evaluation

In [ ]:
# Evaluate model, generate graphs
metrics = model.val(data=data_yaml, imgsz=416)
print("Model accuracy: ", f"{metrics.box.map50 * 100:.2f}%")
print("Model speed (current hardware): ", f"{sum(metrics.speed.values()):.2f}", "ms")

Training images with bounding boxes and confidence levels.

In [ ]:
display.Image("runs/detect/val/val_batch0_pred.jpg")

Precision-confidence curve compares how accurate the model compared with how confident it was in its output.

In [ ]:
display.Image("runs/detect/val/BoxP_curve.png")

Summary of outputs, line should be as strongly diagonal as possible. Off diagonal outputs are misclassifications.

In [ ]:
display.Image("runs/detect/val/confusion_matrix_normalized.png")

# Upload a file to the model

In [ ]:
class ColabFileUploader:
    def __init__(self):
        self.selected = None
        print("Upload a file:")
        uploaded = files.upload()
        if uploaded:
            self.selected = next(iter(uploaded))
            print(f"File '{self.selected}' uploaded.")

fc = ColabFileUploader()

# Runs the model on the uploaded image

In [ ]:
if fc.selected:
    results = model.predict(fc.selected)

    plt.imshow(results[0].plot(pil=True))
    plt.axis('off')
    plt.show()

else:
    print("No file selected.")